In [ ]:
# Load in all the config settings from the 0th python notebook
# %run 0_config.ipynb

# Allow autoreload as we develop the projects/libraries in parallel
%load_ext autoreload
%autoreload 2

%reload_ext autoreload

In [ ]:
import warnings
from pathlib import Path

import seaborn as sns

import matplotlib.pyplot as plt
from urbanopt_des.urbanopt_analysis import URBANoptAnalysis
from urbanopt_des.urbanopt_geojson import DESGeoJSON as URBANoptGeoJSON

sns.set_style("whitegrid")

# Allow autoreload as we develop the projects/libraries in parallel
%load_ext autoreload
%autoreload 2

print("loaded...")

PROJECTED_CRS = "EPSG:3857"

graph_size = (8, 4)

# suppress warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
# The working dir is just the current directory
root_dir = Path().resolve()
ten1_dir = root_dir / "esbe" / "ten1"
uo_analysis_baseline_dir = ten1_dir
uo_analysis_baseline_scenario_name = "baseline_scenario"
uo_analysis_baseline_results = (
    uo_analysis_baseline_dir / "run" / uo_analysis_baseline_scenario_name
)
year = 2017  # do not change... needs to align with day of week.

# Use the ten1 project file and Modelica output location
project_geojson_filename = (
    uo_analysis_baseline_dir / "class_project_ten_coincident.json"
)
des_analysis_agg_dir = ten1_dir / "modelica_project"
results_summary_dir = ten1_dir / "_results_summary"

# Keep these in sync with the overridden project/modelica paths
des_agg_geojson_filename = project_geojson_filename
scenario_file = uo_analysis_baseline_dir / "baseline_scenario.csv"

for dir in [results_summary_dir]:
    if not dir.exists():
        dir.mkdir(parents=True, exist_ok=True)

for required_path in [
    uo_analysis_baseline_dir,
    uo_analysis_baseline_results,
    des_analysis_agg_dir,
    results_summary_dir,
]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required path does not exist: {required_path}")

results_summary_dir.mkdir(parents=True, exist_ok=True)
print(f"Using baseline results from: {uo_analysis_baseline_results}")
print(f"Using modelica results from: {des_analysis_agg_dir}")
print(f"Using project geojson from: {project_geojson_filename}")

In [ ]:
# Read in Modelica-based results
modelica_results, bad_or_empty_results = (
    URBANoptAnalysis.get_list_of_valid_result_folders(des_analysis_agg_dir)
)

# Fallback for projects where results are stored in nested folders (e.g., ten1/modelica_project/*_results/*_res.mat).
if not modelica_results:
    fallback_results = {}
    for mat_path in sorted(des_analysis_agg_dir.rglob("*_res.mat")):
        result_name = mat_path.parent.name
        fallback_results[result_name] = {
            "name": result_name,
            "mat_path": mat_path,
            "error": None,
        }
    if fallback_results:
        modelica_results = fallback_results
        bad_or_empty_results = {}
        print(
            f"Fallback discovered {len(modelica_results)} Modelica result(s) from nested .mat files"
        )

print(f"Found {len(modelica_results)} valid")
for key in modelica_results:
    print(f"  {key}")
print(f"Found {len(bad_or_empty_results)} bad or empty results")
for key, value in bad_or_empty_results.items():
    print(f"  {key} with {value['error']}")

# Optional filter: keep selected model names if they exist; otherwise keep all discovered results.
one_to_keep = []
if one_to_keep:
    print(f"Keeping only the {one_to_keep} modelica results, if they exist")
    modelica_results = {
        key: value for key, value in modelica_results.items() if key in one_to_keep
    }

if not modelica_results:
    raise ValueError(f"No valid modelica results found in {des_analysis_agg_dir}")

# when looking at an individual building result (typically for debugging), use this variable.
# default to the first discovered model if not explicitly set.
model_of_interest = next(iter(modelica_results))
print(f"Model of interest: {model_of_interest}")

## Post Process URBANopt results

In [ ]:
# This GeoJSON file is the one with all of the individual building results in
# order to create an aggregation.
uo_analysis = URBANoptAnalysis(project_geojson_filename, des_analysis_agg_dir, year)

# Currently only one URBANopt result for now
uo_analysis.add_urbanopt_results(
    uo_analysis_baseline_dir, uo_analysis_baseline_scenario_name
)

# Process the building load measure reports.
# Some buildings (for example, houses) may not have simulated load exports yet.
# Treat this as a warning so downstream analysis can continue.
try:
    uo_analysis.urbanopt.process_load_results(uo_analysis.geojson.get_building_ids())
except Exception as exc:
    warnings.warn(
        f"Skipping missing building load exports for now: {exc}", RuntimeWarning
    )

uo_analysis.urbanopt.create_aggregations(uo_analysis.geojson.get_building_ids())

# save the URBANopt dataframes
uo_analysis.urbanopt.save_dataframes()

uo_analysis.urbanopt.display_name = "Non-Connected"

## Process Modelica Results

In [ ]:
# add the analysis from the results search
i = 0
for key, value in modelica_results.items():
    i += 1
    print(f"Adding Modelica results for {value['name']} from {value['mat_path']}")
    uo_analysis.add_modelica_results(value["name"], value["mat_path"])
    # hard code the name to be DES + the iteration if more than
    # one result
    uo_analysis.modelica[key].display_name = f"DES {i}"

    # save the variables from the modelica results
    uo_analysis.modelica[key].save_variables()

uo_analysis.modelica.keys()

In [ ]:
# put the Non-connected result first
# order_of_analyses = ["Non-Connected"] + [key for key in uo_analysis.modelica if key != "Non-Connected"]
# uo_analysis.sort_modelica_results_order(order_of_analyses)

# # get display name mappings - print them in order
# for key, value in uo_analysis.display_name_mappings().items():
#     print(f"  {key} : {value}")


In [ ]:
# We are using an aggregated building model, so read in the GeoJSON of the aggregated building
geojson_agg = URBANoptGeoJSON(des_agg_geojson_filename, skip_validation=True)
print(f"GeoJSON file: {des_agg_geojson_filename}")
print(f"Number of buildings: {len(geojson_agg.get_building_ids())}")

# Align building IDs to what is actually available in the Modelica result.
geojson_building_ids = geojson_agg.get_building_ids()
modelica_building_count = uo_analysis.modelica[model_of_interest].number_of_buildings()
if len(geojson_building_ids) != modelica_building_count:
    warnings.warn(
        (
            "GeoJSON/Modelica building count mismatch "
            f"({len(geojson_building_ids)} vs {modelica_building_count}); "
            "using available Modelica buildings only for now."
        ),
        RuntimeWarning,
    )
building_ids_for_modelica = geojson_building_ids[:modelica_building_count]

# Also store each building's qheat and qcool
other_vars_to_gather = [
    "borFie.Q_flow",
    "TDisWatBorLvg.T",
    "TDisWatRet.T",
    "TDisWatSup.T",
    "TDisWatRet.port_a.m_flow",
    "TDisWatSup.port_a.m_flow",
    "heaPla13e5102e.boiHotWat.TBoiLvg[1]",
    "heaPla13e5102e.boiHotWat.TBoiLvg[2]",
    "heaPla13e5102e.THWRet.T",
    "heaPla13e5102e.THWSup.T",
    "cooPla_67e4a0e1.senMasFlo.m_flow",
    "TimeSerLoa_all_buildings.disFloCoo.PPum",
    "TimeSerLoa_all_buildings.disFloHea.PPum",
    "bui[1].bui.QReqHotWat_flow",
    "bui[1].bui.terUniCoo.mReqChiWat_flow",
    "bui[1].bui.terUniHea.mReqHeaWat_flow",
]

for i in range(1, modelica_building_count + 1):
    other_vars_to_gather.append(f"bui[{i}].bui.QHea_flow")
    other_vars_to_gather.append(f"bui[{i}].bui.QCoo_flow")

uo_analysis.resample_and_convert_modelica_results(
    building_ids_for_modelica, other_vars_to_gather
)
uo_analysis.save_modelica_variables()

# save a copy of the URBANopt OpenStudio results in the analysis directory
uo_analysis.save_urbanopt_results_in_modelica_paths()

# create a combined dataframe for each modelica analysis that includes the
# 60 minute modelica data and the 60 minute openstudio dataframe
uo_analysis.combine_modelica_and_openstudio_results()

# load in the actual data and create the aggregations
uo_analysis.resample_actual_data()

# save the combined data frames for testing
uo_analysis.save_dataframes("min_60_with_buildings")

# aggregations across columns
uo_analysis.create_modelica_aggregations()

# run carbon calculations on the aggregations -
#       **** length of data are not matching up... skipping for now. ***
# uo_analysis.calculate_carbon_emissions("RFCE", 2024, analysis_year=2017, emissions_type="marginal", with_td_losses=True)
# uo_analysis.calculate_carbon_emissions("RFCE", 2045, analysis_year=2017, emissions_type="marginal", with_td_losses=True)

# now roll up to combine rows to monthly, annual, etc.
uo_analysis.create_rollups()

# create the building summary table for URBANopt and each Modelica analysis
uo_analysis.create_building_summaries()

# save the resulting dataframes, which now includes carbon metrics
uo_analysis.save_dataframes()

In [ ]:
uo_analysis.calculate_all_grid_metrics()
# display the grid metric for one of the analysis
# display(uo_analysis.modelica['controlled'].grid_metrics_annual)

# still need to add in utility costs
# uo_analysis.calculate_utility_costs()

# save the dataframes, grid metrics only
uo_analysis.save_dataframes(["grid_metrics_daily", "grid_metrics_annual"])

In [ ]:
# Combine all the results of the analyses for comparison
uo_analysis.create_summary_results()

# print(f"Analysis results are being saved to {uo_analysis.analysis_dir}")
uo_analysis.save_dataframes(["grid_summary", "end_use_summary"])

# create a mapping for the names... but this should go away eventually!
analysis_name_mappings = uo_analysis.display_name_mappings()

In [ ]:
# Create building level results for the UO and Modelica results
buildings_df = uo_analysis.create_building_level_results()
buildings_df.to_csv(
    uo_analysis.urbanopt.scenario_output_path / "building_metrics_annual.csv",
    index=True,
)
print(
    f"Saved to {uo_analysis.urbanopt.scenario_output_path / 'building_metrics_annual.csv'}"
)

display(buildings_df)

In [ ]:
# # REopt - create the CSV data that can be uploaded to REopt
# reopt_df = uo_analysis.urbanopt.data.copy()
# # keep only the 'Total Energy' column
# reopt_df = reopt_df[["Total Building Electricity"]] / 1000  # values need to be in kW
# # rename column 'Total Energy' to 'Loads (kW)'
# reopt_df = reopt_df.rename(columns={"Total Building Electricity": "Loads (kW)"})
# # reset the index which is just the index, i, as it is in hours, drop the index
# reopt_df["Hour"] = range(len(reopt_df))
# reopt_df["Hour"] += 1
# # move the column hour to be first
# reopt_df = reopt_df[["Hour", "Loads (kW)"]]
# # name the index Hour

# display(reopt_df)

# reopt_df.to_csv(uo_analysis.urbanopt.scenario_output_path / "REopt_loads.csv", index=False)

## Post Process the Modelica Results

### Graph Results

In [ ]:
# end use names and colors. The names have to match what is in the end_use_summary results dataframe
energy_end_use_rows = {
    "Interior Lighting": "#FFFFCC",
    "Exterior Lighting": "lightblue",
    "Plug Loads": "brown",
    "Building Cooling": "blue",
    "District Cooling": "blue",
    "District Heating": "orange",
    "Building Heating": "orange",
    "Building Fans": "lightgray",
    "Building Pumps": "lightblue",
    "Building Heat Rejection": "royalblue",
    "Building Water Systems": "#FFBB78",
    "ETS Pump Total": "lightgreen",
    "ETS Heat Pump": "gold",
    "Sewer Pump": "darkgray",
    "GHX Pump": "darkgreen",
    "Distribution Pump": "darkblue",
}

In [ ]:
# Plot only the Non-Connected end uses
# drop all columns except "Units" and "Non-Connected"
df_to_plot = uo_analysis.end_use_summary.copy()
columns_to_drop = [col for col in df_to_plot.columns if col not in ["Non-Connected"]]
df_to_plot = df_to_plot.drop(columns=columns_to_drop)

# remove the rows where the "Non-Connected" values are zero
df_to_plot = df_to_plot[df_to_plot["Non-Connected"] > 0]
# copy the energy_end_use_rows and remove the ones that are not in the df_to_plot
end_use_rows_sub = energy_end_use_rows.copy()
rows_with_data = df_to_plot.index.tolist()
end_use_rows_sub = {
    key: value for key, value in end_use_rows_sub.items() if key in rows_with_data
}

df_to_plot = df_to_plot.T / 1e6
df_to_plot = df_to_plot[end_use_rows_sub.keys()]

# save off the data for sending
df_to_plot.to_csv(results_summary_dir / "end_use_summary_non_connected.csv")
print(f"Saved plot to {results_summary_dir / 'end_use_summary_non_connected.csv'}")

# plot the end use summary with the updated colors
df_to_plot.plot.bar(
    figsize=graph_size, stacked=True, color=end_use_rows_sub.values()
)  # title=f"Analysis Summary"
plt.legend(
    fontsize="x-small", bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.0
)
plt.tight_layout()
plt.subplots_adjust(left=0.1)  # fix the left axis to show the full text
plt.ylabel("Total Energy [MWh]")
# rotate labels on xaxis
plt.xticks(rotation=0)
# add thousands separator to y-axis
plt.gca().get_yaxis().set_major_formatter(
    plt.FuncFormatter(lambda x, _loc: f"{int(x):,}")
)

output_name = results_summary_dir / "bar_end_use_summary_all.png"
plt.savefig(output_name, dpi=300)
plt.show()

In [ ]:
# list analyses
print(["Non-Connected", *list(uo_analysis.modelica.keys())])

analysis_to_exclude = []
analysis_to_exclude_display = [
    analysis_name_mappings[analysis_name] for analysis_name in analysis_to_exclude
]
print(f"Excluding the following analyses: {analysis_to_exclude_display}")

df_to_plot = uo_analysis.end_use_summary.copy()
# remove the columns in analysis_to_exclude_display
df_to_plot = df_to_plot.drop(columns=analysis_to_exclude_display)
df_to_plot = df_to_plot.drop(columns="Units")
print(df_to_plot.columns)
df_to_plot = df_to_plot.T / 1e6
df_to_plot = df_to_plot[energy_end_use_rows.keys()]

# save off the data for sending
df_to_plot.to_csv(results_summary_dir / "end_use_summary_all.csv")
print(f"Saved plot to {results_summary_dir / 'end_use_summary_all.csv'}")


# plot the end use summary with the updated colors
df_to_plot.plot.bar(
    figsize=(9, 6), stacked=True, color=energy_end_use_rows.values()
)  # title=f"Analysis Summary"
plt.legend(fontsize=10, bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.0)
plt.tight_layout()
plt.subplots_adjust(left=0.1)  # fix the left axis to show the full text
plt.ylabel("Total Energy [MWh]")
# rotate labels on xaxis
plt.xticks(rotation=0)
# add thousands separator to y-axis
plt.gca().get_yaxis().set_major_formatter(
    plt.FuncFormatter(lambda x, _loc: f"{int(x):,}")
)
output_name = results_summary_dir / "bar_end_use_summary_all.png"
plt.savefig(output_name, dpi=300)
plt.show()

In [ ]:
# # Plot carbon emissions
# # skip showing the thermal energy in the bar charts and without Individual
# # plot the end use
# emissions_end_use_rows = [
#     "Total Natural Gas Carbon Emissions",
#     "Total Electricity Carbon Emissions 2024",
#     "Total Electricity Carbon Emissions 2045",
#     "Total Carbon Emissions 2024",
#     "Total Carbon Emissions 2045",
# ]

# df_to_plot = uo_analysis.end_use_summary.copy()
# df_to_plot = df_to_plot.drop(columns="Units")
# df_to_plot = df_to_plot.T
# df_to_plot = df_to_plot[emissions_end_use_rows]

# # create carbon type bar chart
# fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
# for name, ax in zip(['2024', '2045'], axes):
#     # create a bar plot for 2024 and 2045 with just the natural gas and electricity
#     columns = [f'Total Natural Gas Carbon Emissions', f'Total Electricity Carbon Emissions {name}']
#     df_to_plot[columns].plot.bar(stacked=True, ax=ax, title=f"Carbon Year {name}")
#     ax.set_xlabel('')
#     ax.set_ylabel('Emissions [mtCO2e]')
#     ax.set_ylim(0, 10000)
#     # rotate x labels to be horizontal
#     ax.tick_params(axis='x', rotation=0)
#     # plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
#     plt.tight_layout()


# # melt the dataframe to plot the emissions over the carbon year
# df_to_plot = df_to_plot[["Total Carbon Emissions 2024", "Total Carbon Emissions 2045"]]
# df_to_plot = df_to_plot.reset_index().melt(id_vars="index", var_name="Carbon Year", value_name="Emissions")
# display(df_to_plot)
# fig, ax = plt.subplots(figsize=(10, 5))
# sns.barplot(df_to_plot, x="index", y="Emissions", hue="Carbon Year", ax=ax)
# ax.set_xlabel("")
# ax.set_ylabel("Carbon Emissions [mtCO2e/year]")
# # add thousand separators in y axis
# ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, _loc: f"{int(x):,}"))

# plt.legend(title=None)
# plt.tight_layout()
# plt.savefig(results_summary_dir / "bar_end_use_summary_emissions.png", dpi=300)
# plt.show()

In [ ]:
# Create two subplots in one figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

df_to_plot = uo_analysis.modelica[model_of_interest].min_60_with_buildings

# Top plot: Total ETS Pump Electricity
df_top = (
    df_to_plot[["ETS Pump Electricity Building all_buildings"]] / 1000
)  # convert to kW
df_top.plot(ax=ax1, color="black", linewidth=0.5, legend=False)
ax1.set_ylabel("Power [kW]")
ax1.set_title("ETS Pump Electricity")
ax1.grid(True, alpha=0.3)

# Bottom plot: HHW and CHW
df_bottom = (
    df_to_plot[
        [
            "ETS Pump HHW Electricity Building all_buildings",
            "ETS Pump CHW Electricity Building all_buildings",
        ]
    ]
    / 1000
)  # convert to kW
df_bottom.plot(ax=ax2, color=["red", "blue"], linewidth=0.5)
ax2.set_ylabel("Power [kW]")
ax2.set_xlabel("Time")
ax2.set_title("ETS Pump HHW and CHW Electricity")
ax2.legend(["HHW", "CHW"], loc="upper right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot the requested load vs the supplied load
df_to_plot = uo_analysis.modelica[model_of_interest].min_60_with_buildings
df_to_plot = df_to_plot[
    ["bui[1].bui.terUniHea.mReqHeaWat_flow", "bui[1].bui.terUniCoo.mReqChiWat_flow"]
]
df_to_plot = df_to_plot / 1000  # convert to kW
# normalize the data

display(df_to_plot)
df_to_plot.plot(figsize=(10, 5), color=["red", "blue"], alpha=0.5, linewidth=0.3)
plt.ylabel("Massflow [kg/s]")
plt.xlabel("Time")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=2)
plt.tight_layout()

In [ ]:
# seasonal plots - list of vars to plat
for analysis_name in ["Non-Connected", *list(uo_analysis.modelica.keys())]:
    vars_to_plot = [
        "Total Electricity",
        "Total Natural Gas",
        # "Total Thermal Cooling Energy",
        # "Total Thermal Heating Energy",
        # "District Loop Energy",
        "Total Energy",
    ]

    if analysis_name == "Non-Connected":
        df_season = uo_analysis.urbanopt.data
    else:
        df_season = uo_analysis.modelica[analysis_name].min_60_with_buildings

    for var_to_plot in vars_to_plot:
        str_analysis_name = analysis_name.lower().replace(" ", "_")
        str_vars_to_plot = var_to_plot.lower().replace(" ", "_")

        # skip the plot if all the data are zero
        if df_season[var_to_plot].sum() == 0:
            print(
                f"Skipping plot for {analysis_name} - {var_to_plot} because the sum is zero"
            )
            continue

        # remove the first two data of the data, the warm up peak is annoying!
        df_season = df_season.iloc[48:]

        # Monthly consumption (in MWh)
        var_y_min = df_season[var_to_plot].min() / 1e6
        var_y_max = df_season[var_to_plot].max() / 1e6

        # extend past the min / max by 20% for plotting
        # if var_y_min + var_y_max is < 0, then extend the min and max in opposite directions
        if var_y_min + var_y_max < 0:
            var_y_min = var_y_min * 1.2
            var_y_max = var_y_max * 0.8
        else:
            var_y_min = var_y_min * 0.8
            var_y_max = var_y_max * 1.2

        # if the min or max is so close to 0, then just make zero
        if abs(var_y_min) < 1 and abs(var_y_max) < 1:
            # do nothing
            pass
        else:
            if abs(var_y_min) < 1:
                var_y_min = 0
            if abs(var_y_max) < 1:
                var_y_max = 0

        _fig, ax = plt.subplots(figsize=graph_size)
        sns.boxplot(data=df_season / 1e6, x=df_season.index.month, y=var_to_plot, ax=ax)
        ax.set_ylabel(f"{var_to_plot} (MW)")
        # ax.set_title(f"{analysis_name_mappings[analysis_name]} - {var_to_plot}")
        ax.set_xlabel("Month")
        ax.set_ylim(var_y_min, var_y_max)
        plt.title(analysis_name)
        plt.tight_layout()
        output_name = (
            results_summary_dir
            / f"monthly_boxplot_{str_vars_to_plot}_{str_analysis_name}.png"
        )
        plt.savefig(output_name, dpi=300)

        # Weekday consumption
        _fig, ax = plt.subplots(figsize=graph_size)
        sns.boxplot(
            data=df_season / 1e6, x=df_season.index.day_name(), y=var_to_plot, ax=ax
        )
        ax.set_ylabel(f"{var_to_plot} (MW)")
        # ax.set_title(f"{analysis_name_mappings[analysis_name]} - {var_to_plot}")
        # ax.set_xlabel('Weekday')
        ax.set_ylim(var_y_min, var_y_max)
        plt.title(analysis_name)
        plt.tight_layout()
        output_name = (
            results_summary_dir
            / f"weekday_boxplot_{str_vars_to_plot}_{str_analysis_name}.png"
        )
        plt.savefig(output_name, dpi=300)

In [ ]:
# Combined boxplots for energy usage monthly
import pandas as pd

# seasonal plots - list of vars to plat
for analysis_name in ["Non-Connected", *list(uo_analysis.modelica.keys())]:
    vars_to_plot = [
        "Total Electricity",
        "Total Natural Gas",
        # "Total Thermal Cooling Energy",
        # "Total Thermal Heating Energy",
        # "District Loop Energy",
        "Total Energy",
    ]

for var_to_plot in vars_to_plot:
    combined_df = []

    for analysis_name in ["Non-Connected", *list(uo_analysis.modelica.keys())]:
        if analysis_name == "Non-Connected":
            df_season = uo_analysis.urbanopt.data.copy() / 1e6
            df_season["Analysis"] = analysis_name
        else:
            df_season = (
                uo_analysis.modelica[analysis_name].min_60_with_buildings.copy() / 1e6
            )
            df_season["Analysis"] = uo_analysis.modelica[analysis_name].display_name

        df_season = df_season.iloc[48:]  # Remove initial warm-up peak
        combined_df.append(df_season)

    combined_df = pd.concat(combined_df)

    # Monthly consumption
    _fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(
        data=combined_df,
        x=combined_df.index.month,
        y=var_to_plot,
        hue="Analysis",
        dodge=True,
        width=0.7,
        ax=ax,
    )

    var_y_min = 0
    var_y_max = combined_df[var_to_plot].max() * 1.2

    ax.set_ylabel(f"{var_to_plot} (MW)")
    ax.set_xlabel("Month")
    ax.set_ylim(var_y_min, var_y_max)
    # plt.legend(title="Analysis")
    plt.tight_layout()
    plt.savefig(
        results_summary_dir
        / f"monthly_boxplot_{var_to_plot.lower().replace(' ', '_')}.png",
        dpi=300,
    )
    plt.show()


# TODO: And annual boxplots???

In [ ]:
# daily box plots of grid vars
vars_to_plot = [
    "Total Electricity Max",
    "Total Electricity Load Factor",
    "Total Electricity System Ramping",
    "Total Electricity PVR",
    # "Total Building Natural Gas Max",
    # "Total Building Natural Gas Load Factor",
    "Total Natural Gas Max",
    "Total Natural Gas Load Factor",
    "Total Natural Gas System Ramping",
    # 'Total Thermal Cooling Energy Load Factor',
    # 'Total Thermal Heating Energy Load Factor',
    # 'Total Thermal Cooling Energy System Ramping',
    # 'Total Thermal Heating Energy System Ramping',
    # 'District Loop Energy Load Factor',
    # 'District Loop Energy System Ramping',
]

for analysis_name in ["Non-Connected", *list(uo_analysis.modelica.keys())]:
    if analysis_name == "Non-Connected":
        df_box = uo_analysis.urbanopt.grid_metrics_daily
    else:
        df_box = uo_analysis.modelica[analysis_name].grid_metrics_daily

    for var_to_plot in vars_to_plot:
        y_min = df_box[var_to_plot].min() * 0.8
        y_max = df_box[var_to_plot].max() * 1.2
        # Monthly consumption
        _fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(data=df_box, x=df_box.index.month, y=var_to_plot, ax=ax)
        if "System Ramping" in var_to_plot:
            ax.set_ylabel(f"{var_to_plot} (MW)")
            save_name = "system_ramping"
            # grab the y_max from the data and round to the ceiling
        elif "Load Factor" in var_to_plot:
            ax.set_ylabel(f"{var_to_plot} (Ratio)")
            save_name = "load_factor"
            y_min = 0
            y_max = 1
        else:
            ax.set_ylabel(f"{var_to_plot} (W)")
            save_name = "electricity"
        # ax.set_title(f"Daily {var_to_plot} for {analysis_name_mappings[analysis_name]}")
        ax.set_xlabel("Month")

        var_to_plot_str = var_to_plot.lower().replace(" ", "_")
        if y_max:
            if y_min:
                ax.set_ylim(y_min, y_max)
            else:
                ax.set_ylim(0, y_max)
        plt.title(analysis_name)
        plt.tight_layout()
        plt.savefig(
            results_summary_dir
            / f"monthly_grid_metric_{var_to_plot_str}_{analysis_name}.png",
            dpi=300,
        )
        plt.show()

        # Weekday consumption
        y_min = df_box[var_to_plot].min() * 0.8
        y_max = df_box[var_to_plot].max() * 1.2
        _fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(
            data=df_box, x=df_box.index.day_name(), y=var_to_plot, ax=ax
        )  # ,  order=['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday'])
        # resort the data to be sunday first
        if "System Ramping" in var_to_plot:
            ax.set_ylabel(f"{var_to_plot} (MW)")
            save_name = "system_ramping"
        elif "Load Factor" in var_to_plot:
            ax.set_ylabel(f"{var_to_plot} (Ratio)")
            save_name = "load_factor"
            y_min = 0
            y_max = 1
        else:
            ax.set_ylabel(f"{var_to_plot} (W)")
            save_name = "electricity"
        ax.set_xlabel("Day of Week")

        var_to_plot_str = var_to_plot.lower().replace(" ", "_")
        if y_max:
            if y_min:
                ax.set_ylim(y_min, y_max)
            else:
                ax.set_ylim(0, y_max)
        plt.title(analysis_name)
        plt.tight_layout()
        plt.savefig(
            results_summary_dir
            / f"weekday_grid_metric_{var_to_plot_str}_{analysis_name}.png",
            dpi=300,
        )
        plt.show()

In [ ]:
# Creating combined boxplots
import pandas as pd

# daily box plots of grid vars
vars_to_plot = [
    "Total Electricity Max",
    "Total Electricity Load Factor",
    "Total Electricity System Ramping",
    "Total Electricity PVR",
    # "Total Building Natural Gas Max",
    # "Total Building Natural Gas Load Factor",
]
for var_to_plot in vars_to_plot:
    combined_df = []  # Empty dataframe to hold all the analysis options

    for analysis_name in ["Non-Connected", *list(uo_analysis.modelica.keys())]:
        if analysis_name == "Non-Connected":
            df_box = uo_analysis.urbanopt.grid_metrics_daily
            df_box["Analysis"] = analysis_name

        else:
            df_box = uo_analysis.modelica[analysis_name].grid_metrics_daily
            df_box["Analysis"] = uo_analysis.modelica[analysis_name].display_name

        combined_df.append(df_box)

    combined_df = pd.concat(combined_df)

    # Monthly consumption
    _fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(
        data=combined_df,
        x=combined_df.index.month,
        y=var_to_plot,
        hue="Analysis",
        dodge=True,
        width=0.7,
        ax=ax,
    )
    if "System Ramping" in var_to_plot:
        ax.set_ylabel(f"{var_to_plot} (MW)")
        y_min, y_max = 0, combined_df[var_to_plot].max() * 1.2
    elif ("Load Factor") in var_to_plot:
        ax.set_ylabel(f"{var_to_plot} (Ratio)")
        y_min, y_max = 0.4, 0.9
    elif ("PVR") in var_to_plot:
        ax.set_ylabel(f"{var_to_plot}")
        y_min, y_max = 0, combined_df[var_to_plot].max() * 1.2
    else:
        ax.set_ylabel(f"{var_to_plot} (W)")  # TODO: Plot in MW
        y_min, y_max = 0, combined_df[var_to_plot].max() * 1.2

    ax.set_xlabel("Month")
    ax.set_ylim(y_min, y_max)
    plt.tight_layout()
    var_to_plot_str = var_to_plot.lower().replace(" ", "_")
    plt.savefig(
        results_summary_dir
        / f"monthly_grid_metric_{var_to_plot_str}_{analysis_name}.png",
        dpi=300,
    )
    plt.show()

    # Define correct day order starting with Sunday
    day_order = [
        "Sunday",
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
    ]

    # Weekday consumption
    _fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(
        data=combined_df,
        x=combined_df.index.day_name(),
        y=var_to_plot,
        hue="Analysis",
        ax=ax,
        order=day_order,
    )
    if "System Ramping" in var_to_plot:
        ax.set_ylabel(f"{var_to_plot} (MW)")
        y_min, y_max = 0, combined_df[var_to_plot].max() * 1.2
    elif ("Load Factor") in var_to_plot:
        ax.set_ylabel(f"{var_to_plot} (Ratio)")
        y_min, y_max = 0.4, 0.9
    elif ("PVR") in var_to_plot:
        ax.set_ylabel(f"{var_to_plot}")
        y_min, y_max = 0, combined_df[var_to_plot].max() * 1.2
    else:
        ax.set_ylabel(f"{var_to_plot} (W)")  # TODO: Plot in MW
        y_min, y_max = 0, combined_df[var_to_plot].max() * 1.2

    ax.set_xlabel("Day of Week")
    ax.set_ylim(y_min, y_max)
    plt.tight_layout()
    var_to_plot_str = var_to_plot.lower().replace(" ", "_")
    plt.savefig(
        results_summary_dir
        / f"weekday_grid_metric_{var_to_plot_str}_{analysis_name}.png",
        dpi=300,
    )
    plt.show()

In [ ]:
## Post-processing and figures related to combined building thermal loads

# Pull in aggregate loads timeseries file
from pathlib import Path

import matplotlib.dates as mdates
import pandas as pd

cur_file = Path.cwd()

# Make the path finding more robust - search for the directory
loads_loc = None

# Search in both analysis and analysis_v1 directories
for base_dir in ["analysis", "analysis_v1"]:
    search_path = (
        cur_file
        / base_dir
        / "urbanopt"
        / "flxenabler"
        / "run"
        / "baseline"
        / "all_buildings"
        / "01_export_modelica_loads"
    )
    if search_path.exists():
        loads_loc = search_path
        print(f"Found loads directory at: {loads_loc}")
        break

if loads_loc is None:
    print(
        "Could not find 01_export_modelica_loads directory in analysis or analysis_v1"
    )
    # Try to find it recursively as a fallback
    for analysis_dir in cur_file.glob(
        "analysis*/urbanopt/*/run/baseline/all_buildings/01_export_modelica_loads"
    ):
        loads_loc = analysis_dir
        print(f"Found loads directory at: {loads_loc}")
        break

if loads_loc is None or not (loads_loc / "building_loads.csv").exists():
    print("ERROR: Could not find building_loads.csv file")
else:
    df_agg_loads = pd.read_csv(
        loads_loc / "building_loads.csv",
        header=0,
        names=["Date", "Cooling", "Heating", "SWH"],
    )  # Watts
    df_agg_loads_2 = df_agg_loads[["Cooling", "Heating"]] / 1000  # kW
    df_agg_loads_2 = df_agg_loads_2.set_index(pd.to_datetime(df_agg_loads["Date"]))

    # Save some basic stats
    df_stats = pd.Series()
    df_stats["peak_cooling"] = df_agg_loads["Cooling"].min() / 1e6  # MW
    df_stats["peak_heating"] = df_agg_loads["Heating"].max() / 1e6  # MW

    df_stats["total_cooling"] = df_agg_loads["Cooling"].sum() / 1e6  # MWh
    df_stats["total_heating"] = df_agg_loads["Heating"].sum() / 1e6  # MWh

    df_stats.to_csv(results_summary_dir / "building_loads_summary.csv")
    print(
        f"Saved building loads summary to {results_summary_dir / 'building_loads_summary.csv'}"
    )
    print(df_stats)

    # Print the timeseries aggregate loads
    _fig, ax = plt.subplots(figsize=(8, 5))
    sns.lineplot(data=df_agg_loads_2, ax=ax, alpha=0.5, linewidth=0.5)
    ax.set_ylabel("Aggregate thermal load (kW)")
    ax.set_xlabel("Time")
    ax.grid(False)
    # ax.set_ylim(y_min, y_max)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    plt.tight_layout()
    plt.savefig(results_summary_dir / "aggregate_building_loads.png", dpi=300)
    plt.show()

In [ ]:
## Plot a week of timeseries electrical loads for all 3 systems
# Having trouble pulling this programatically from variables, it seems like the 4G and 5G system might be overwriting each other for some of the variables?
# So just pulling manually from the output files for now.

# Import the weather file
from pathlib import Path

# name of the 2 models to plot and their respective directories
models_to_plot = {
    # plot name : directory name
    "4G DES": "fourg_system",
    "5G DES HPTrio": "fiveg_hptrio_controlled_flow",
}

weather_candidates = sorted((uo_analysis_baseline_dir / "weather").glob("*.mos"))
if not weather_candidates:
    raise FileNotFoundError(
        f"No .mos weather file found in {uo_analysis_baseline_dir / 'weather'}"
    )
weather_loc = weather_candidates[0]
print(f"Using weather file: {weather_loc.name}")
df_weather = pd.read_csv(weather_loc, delim_whitespace=True, skiprows=40, header=None)
temps = df_weather.iloc[:, 1]  # Dry bulb temp in C

# Pull together the correct rows into a dataframe for whole year
combined_df = []  # Empty dataframe to hold all the analysis options

df_to_plot = (
    pd.DataFrame()
)  # Putting "total electricity" from each district into a single dataframe
df_to_plot["Non-Connected"] = uo_analysis.urbanopt.data["Total Electricity"]

df_in = pd.read_csv(
    des_analysis_agg_dir / "fourg_system" / "power_60min_with_buildings.csv",
    parse_dates=["datetime"],
    index_col="datetime",
)
df_to_plot["4G DES"] = df_in["Total Electricity"]

for model_name, model_dir in models_to_plot.items():
    df_in = pd.read_csv(
        des_analysis_agg_dir / model_dir / "power_60min_with_buildings.csv",
        parse_dates=["datetime"],
        index_col="datetime",
    )
    df_to_plot[model_name] = df_in["Total Electricity"]

df_to_plot = df_to_plot / 1e6  # Convert to MW
df_to_plot["Temperature"] = temps.to_numpy()

## Summer:
start_datetime = pd.Timestamp(
    "2017-07-16 00:00:00"
)  # Approximately a Sunday. July 16 2017?
end_datetime = start_datetime + pd.Timedelta(weeks=1)
# Filter for only the desired date range (one week window)
df_to_plot_summer = df_to_plot[
    (df_to_plot.index >= start_datetime) & (df_to_plot.index < end_datetime)
]

# Plot
plt.rcParams.update({"font.size": 12})  # Adjust this as needed
_fig, ax1 = plt.subplots(figsize=(9, 5))
df_to_plot_summer_small = df_to_plot_summer.drop("Temperature", axis=1)
sns.lineplot(data=df_to_plot_summer_small, ax=ax1)
y_min, y_max = -2, 5
ax1.set_ylabel("Electricity demand (MW)")
ax1.set_ylim(y_min, y_max)

# Adding temperature
df_to_plot_summer_small = df_to_plot_summer["Temperature"]
# Second y-axis (temperature)
ax2 = ax1.twinx()
sns.lineplot(data=df_to_plot_summer_small, ax=ax2, color="k")
ax2.grid(False)
ax2.set_ylim(20, 70)
ax2.set_ylabel("Temperature (°C)")
ax2.set_xlabel("Time (hours)")

plt.tight_layout()
# plt.savefig(results_summary_dir / f"summer_week_{var_to_plot_str}_{analysis_name}.png", dpi=300)
plt.savefig(results_summary_dir / f"summer_week_{analysis_name}.svg")
plt.show()

## Winter:
start_datetime = pd.Timestamp("2017-01-08 00:00:00")  # Sunday.
end_datetime = start_datetime + pd.Timedelta(weeks=1)
# Filter for only the desired date range (one week window)
df_to_plot_winter = df_to_plot[
    (df_to_plot.index >= start_datetime) & (df_to_plot.index < end_datetime)
]

# Plot
_fig, ax1 = plt.subplots(figsize=(9, 5))
df_to_plot_summer_small = df_to_plot_winter.drop("Temperature", axis=1)
sns.lineplot(data=df_to_plot_summer_small, ax=ax1)
y_min, y_max = -2, 5
ax1.set_ylabel("Electricity demand (MW)")
ax1.set_ylim(y_min, y_max)

# Adding temperature
df_to_plot_summer_small = df_to_plot_winter["Temperature"]
ax2 = ax1.twinx()
sns.lineplot(data=df_to_plot_summer_small, ax=ax2, color="k")
ax2.grid(False)
ax2.set_ylim(-10, 40)
ax2.set_ylabel("Temperature (°C)")
ax2.set_xlabel("Time (hours)")

plt.tight_layout()
var_to_plot_str = var_to_plot.lower().replace(" ", "_")
# plt.savefig(results_summary_dir / f"winter_week_{var_to_plot_str}_{analysis_name}.png", dpi=300)
plt.savefig(results_summary_dir / f"winter_week_{analysis_name}.svg")
plt.show()

## Spring:
start_datetime = pd.Timestamp("2017-04-16 00:00:00")  # Sunday.
end_datetime = start_datetime + pd.Timedelta(weeks=1)
# Filter for only the desired date range (one week window)
df_to_plot_spring = df_to_plot[
    (df_to_plot.index >= start_datetime) & (df_to_plot.index < end_datetime)
]

# Plot
_fig, ax1 = plt.subplots(figsize=(9, 5))
df_to_plot_summer_small = df_to_plot_spring.drop("Temperature", axis=1)
sns.lineplot(data=df_to_plot_summer_small, ax=ax1)
y_min, y_max = -2, 5
ax1.set_ylabel("Electricity demand (MW)")
ax1.set_ylim(y_min, y_max)

# Adding temperature
df_to_plot_summer_small = df_to_plot_spring["Temperature"]
ax2 = ax1.twinx()
sns.lineplot(data=df_to_plot_summer_small, ax=ax2, color="k")
ax2.grid(False)
ax2.set_ylim(0, 60)
ax2.set_ylabel("Temperature (°C)")
ax2.set_xlabel("Time (hours)")

plt.tight_layout()
var_to_plot_str = var_to_plot.lower().replace(" ", "_")
# plt.savefig(results_summary_dir / f"spring_week_{var_to_plot_str}_{analysis_name}.png", dpi=300)
plt.savefig(results_summary_dir / f"spring_week_{analysis_name}.svg")
plt.show()

In [ ]:
## Plotting load duration curves for Total Electricity and Total Natural Gas
# Note that Total Natural Gas does not include the DES boiler at the moment...

# Load Duration Curves (LDCs)
ldc_vars = ["Total Electricity", "Total Natural Gas", "Total Energy"]

for ldc_var in ldc_vars:
    combined_ldc_df = []  # To hold LDC data from all scenarios
    combined_ldc_df_100 = []  # To hold LDC data from all scenarios

    for analysis_name in ["Non-Connected", *list(uo_analysis.modelica.keys())]:
        if analysis_name == "Non-Connected":
            df_15min = uo_analysis.urbanopt.data_15min[ldc_var].copy()
            df_hourly = df_15min.resample("H").max()  # Resample to hourly max
            display_name = "Non-Connected"
        else:
            df_15min = (
                uo_analysis.modelica[analysis_name]
                .min_15_with_buildings[ldc_var]
                .copy()
            )
            df_hourly = df_15min.resample("H").max()  # Resample to hourly max
            display_name = uo_analysis.modelica[analysis_name].display_name

        # Sort hourly load descending for LDC
        ldc_series = df_hourly.sort_values(ascending=False).reset_index(drop=True)
        ldc_df = pd.DataFrame(
            {
                "Hour Rank": range(1, len(ldc_series) + 1),
                "Load (W)": ldc_series,
                "Analysis": display_name,
            }
        )
        combined_ldc_df.append(ldc_df)
        combined_ldc_df_100.append(ldc_df.head(100))

    combined_ldc_df = pd.concat(combined_ldc_df)
    combined_ldc_df_100 = pd.concat(combined_ldc_df_100)
    print(f"Processing LDC for {combined_ldc_df}")
    print(f"Processing LDC 100 for {combined_ldc_df_100}")

    # Plot LDC
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.lineplot(
        data=combined_ldc_df, x="Hour Rank", y="Load (W)", hue="Analysis", ax=ax
    )
    ax.set_ylabel(f"{ldc_var} Load (W)")
    ax.set_xlabel("Hour of Year (Sorted by Load)")
    ax.set_xlim(0, len(ldc_series) + 20)
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()

    # Save with appropriate filename
    filename = f"load_duration_curve_{ldc_var.lower().replace(' ', '_')}.png"
    plt.savefig(results_summary_dir / filename, dpi=300)
    print(f"Saved {filename}")
    plt.show()

    # zoom into the first 100 hours and plot
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.lineplot(
        data=combined_ldc_df_100, x="Hour Rank", y="Load (W)", hue="Analysis", ax=ax
    )
    ax.set_ylabel(f"{ldc_var} Load (W)")
    ax.set_xlabel("Hour of Year (Sorted by Load)")
    ax.set_xlim(0, 100)
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()

    # Save zoomed plot with appropriate filename
    filename = f"load_duration_curve_{ldc_var.lower().replace(' ', '_')}_first100.png"
    plt.savefig(results_summary_dir / filename, dpi=300)
    print(f"Saved {filename}")
    plt.show()